<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_09_3_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 9: Foundations of Generative AI**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 9 Material

* Part 9.1: Transformer Architecture [[Video]]() [[Notebook]](t81_558_class_09_1_transformers.ipynb)
* Part 9.2: Pretrained Models and the Hugging Face Ecosystem [[Video]]() [[Notebook]](t81_558_class_09_2_hugging.ipynb)
* **Part 9.3: Embeddings and Vector Representations** [[Video]]() [[Notebook]](t81_558_class_09_3_embedding.ipynb)
* Part 9.4: Diffusion Models and Image Generation [[Video]]() [[Notebook]](t81_558_class_09_4_stable_diff.ipynb)
* Part 9.5: Accessing API-Based LLMs [[Video]]() [[Notebook]](t81_558_class_09_5_chat_gpt.ipynb)


# Google CoLab Instructions

The following code detects whether the notebook is running in Google CoLab.

In [1]:
try:
    from google.colab import drive
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

Note: using Google CoLab


# Part 9.3: Embeddings and Vector Representations

[Embedding Layers](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) are a handy feature of PyTorch that allows the program to automatically insert additional information into the data flow of your neural network. An embedding layer would automatically allow you to insert vectors in the place of word indexes.  

Programmers often use embedding layers with Natural Language Processing (NLP); however, you can use these layers when you wish to insert a lengthier vector in an index value place. In some ways, you can think of an embedding layer as dimension expansion. However, the hope is that these additional dimensions provide more information to the model and provide a better score.

## Simple Embedding Layer Example

* **num_embeddings** = How large is the vocabulary?  How many categories are you encoding? This parameter is the number of items in your "lookup table."
* **embedding_dim** = How many numbers in the vector you wish to return.

Now we create a neural network with a vocabulary size of 10, which will reduce those values between 0-9 to 4 number vectors. This neural network does nothing more than passing the embedding on to the output. But it does let us see what the embedding is doing. Each feature vector coming in will have two such features.

In [2]:
import torch
import torch.nn as nn

embedding_layer = nn.Embedding(num_embeddings=10, embedding_dim=4)
optimizer = torch.optim.Adam(embedding_layer.parameters(), lr=0.001)
loss_function = nn.MSELoss()

Let's take a look at the structure of this neural network to see what is happening inside it.

In [3]:
print(embedding_layer)

Embedding(10, 4)


For this neural network, which is just an embedding layer, the input is a vector of size 2. These two inputs are integer numbers from 0 to 9 (corresponding to the requested input_dim quantity of 10 values). Looking at the summary above, we see that the embedding layer has 40 parameters. This value comes from the embedded lookup table that contains four amounts (output_dim) for each of the 10 (input_dim) possible integer values for the two inputs. The output is 2 (input_length) length 4 (output_dim) vectors, resulting in a total output size of 8, which corresponds to the Output Shape given in the summary above.

Now, let us query the neural network with two rows. The input is two integer values, as was specified when we created the neural network.

In [4]:
input_tensor = torch.tensor([[1, 2]], dtype=torch.long)
pred = embedding_layer(input_tensor)

print(input_tensor.shape)
print(pred)


torch.Size([1, 2])
tensor([[[ 0.9680, -1.7282,  0.2045, -0.6579],
         [ 1.0805,  2.3466,  0.6005,  2.3663]]], grad_fn=<EmbeddingBackward0>)


Here we see two length-4 vectors that PyTorch looked up for each input integer. Recall that Python arrays are zero-based. PyTorch replaced the value of 1 with the second row of the 10 x 4 lookup matrix. Similarly, PyTorch returned the value of 2 by the third row of the lookup matrix. The following code displays the lookup matrix in its entirety. The embedding layer performs no mathematical operations other than inserting the correct row from the lookup table.

In [5]:
embedding_layer.weight.data

tensor([[ 1.5427,  0.6920, -0.5971,  1.1490],
        [ 0.9680, -1.7282,  0.2045, -0.6579],
        [ 1.0805,  2.3466,  0.6005,  2.3663],
        [-0.5293, -1.6077, -0.0614,  1.1960],
        [ 0.3575,  0.3307,  0.2279, -1.1295],
        [-0.5499, -0.0053, -0.3140,  0.6443],
        [-0.4358,  0.3898, -2.1413,  1.2849],
        [ 0.6338,  2.8180, -0.3198, -0.1937],
        [-0.5686,  0.8120,  0.9030, -0.6162],
        [-0.8495,  0.8428,  1.8175,  0.6187]])

The values above are random parameters that PyTorch generated as starting points.  Generally, we will transfer an embedding or train these random values into something useful.  The following section demonstrates how to embed a hand-coded embedding.

## Transferring An Embedding

Now, we see how to hard-code an embedding lookup that performs a simple one-hot encoding.  One-hot encoding would transform the input integer values of 0, 1, and 2 to the vectors $[1,0,0]$, $[0,1,0]$, and $[0,0,1]$ respectively. The following code replaced the random lookup values in the embedding layer with this one-hot coding-inspired lookup table.

In [6]:
import torch
import torch.nn as nn

# Define the embedding lookup matrix
embedding_lookup = torch.tensor([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
], dtype=torch.float32)  # Make sure to use float32 for weight matrices

# Create the embedding layer
embedding_layer = nn.Embedding(num_embeddings=3, embedding_dim=3)

# Set the weights of the embedding layer
embedding_layer.weight.data = embedding_lookup


We have the following parameters for the Embedding layer:
    
* input_dim=3 - There are three different integer categorical values allowed.
* output_dim=3 - Three columns represent a categorical value with three possible values per one-hot encoding.
* input_length=2 - The input vector has two of these categorical values.

We query the neural network with two categorical values to see the lookup performed.

In [7]:
# Create the input tensor directly in PyTorch
input_tensor = torch.tensor([[0, 1]], dtype=torch.long)

# Forward pass to get the predictions
pred = embedding_layer(input_tensor)

print(input_tensor.shape)
print(pred)

torch.Size([1, 2])
tensor([[[1., 0., 0.],
         [0., 1., 0.]]], grad_fn=<EmbeddingBackward0>)


The given output shows that we provided the program with two rows from the one-hot encoding table. This encoding is a correct one-hot encoding for the values 0 and 1, where there are up to 3 unique values possible.

The following section demonstrates how to train this embedding lookup table.

## Training an Embedding

First, we make use of the following imports.

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim


We create a neural network that classifies restaurant reviews according to positive or negative.  This neural network can accept strings as input, such as given here.  This code also includes positive or negative labels for each review.

In [9]:
# Define 10 restaurant reviews.
reviews = [
    'Never coming back!',
    'Horrible service',
    'Rude waitress',
    'Cold food.',
    'Horrible food!',
    'Awesome',
    'Awesome service!',
    'Rocks!',
    'poor work',
    'Couldn\'t have done better']

# Define labels (1=negative, 0=positive)
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]


Notice that the second to the last label is incorrect. Errors such as this are not too out of the ordinary, as most training data could have some noise.

We define a vocabulary size of 50. Though we do not have 50 distinct words, it is okay to use a value larger than needed. For input, we convert each word to an integer index by hashing the word and taking the result modulo the vocabulary size. This is a simple, stateless way to map words to indices, but note that different words can collide on the same index when the vocabulary is small. In a production setting, you would typically build a proper vocabulary from your training data and assign each unique word its own index.

In [10]:
# Encode each word as an integer index in [0, VOCAB_SIZE)
VOCAB_SIZE = 50
encoded_reviews = [torch.tensor([hash(word) % VOCAB_SIZE for word in review.split()]) for review in reviews]

print(f"Encoded reviews: {encoded_reviews}")


Encoded reviews: [tensor([ 6, 49, 36]), tensor([ 1, 42]), tensor([45, 32]), tensor([29,  7]), tensor([ 1, 19]), tensor([8]), tensor([ 8, 18]), tensor([31]), tensor([47, 13]), tensor([35, 16, 14, 43])]


The code above converts each review to a sequence of word indexes; however, the resulting sequences have different lengths. We pad the shorter reviews to length 4 and truncate any words beyond the fourth word.

In [11]:
MAX_LENGTH = 4

# Pad each review to MAX_LENGTH (truncating longer reviews, padding shorter ones with 0)
def pad_or_truncate(seq, length):
    seq = seq[:length]
    if len(seq) < length:
        seq = torch.cat([seq, torch.zeros(length - len(seq), dtype=seq.dtype)])
    return seq

padded_reviews = torch.stack([pad_or_truncate(r, MAX_LENGTH) for r in encoded_reviews])
print(padded_reviews)


tensor([[ 6, 49, 36,  0],
        [ 1, 42,  0,  0],
        [45, 32,  0,  0],
        [29,  7,  0,  0],
        [ 1, 19,  0,  0],
        [ 8,  0,  0,  0],
        [ 8, 18,  0,  0],
        [31,  0,  0,  0],
        [47, 13,  0,  0],
        [35, 16, 14, 43]])


Each review is padded by appending zeros at the end so that every sequence has the same length.

Next, we create a neural network to learn to classify these reviews.

In [12]:
model = nn.Sequential(
    nn.Embedding(VOCAB_SIZE, 8),
    nn.Flatten(),
    nn.Linear(8 * MAX_LENGTH, 1),
    nn.Sigmoid()
)

This network accepts four integer inputs that specify the indexes of a padded movie review. The first embedding layer converts these four indexes into four length vectors 8. These vectors come from the lookup table that contains 50 (VOCAB_SIZE) rows of vectors of length 8. This encoding is evident by the 400 (8 times 50) parameters in the embedding layer. The output size from the embedding layer is 32 (4 words expressed as 8-number embedded vectors). A single output neuron is connected to the embedding layer by 33 weights (32 from the embedding layer and a single bias neuron). Because this is a single-class classification network, we use the sigmoid activation function and binary_crossentropy.

The program now trains the neural network. The embedding lookup and dense 33 weights are updated to produce a better score.

In [13]:
criterion = nn.BCELoss()  # Binary Cross Entropy
optimizer = optim.Adam(model.parameters())

# Training the model
epochs = 100
labels_tensor = torch.tensor(labels, dtype=torch.float)
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(padded_reviews.long())
    loss = criterion(outputs.squeeze(), labels_tensor)
    loss.backward()
    optimizer.step()


We can see the learned embeddings.  Think of each word's vector as a location in the 8 dimension space where words associated with positive reviews are close to other words.  Similarly, training places negative reviews close to each other.  In addition to the training setting these embeddings, the 33 weights between the embedding layer and output neuron similarly learn to transform these embeddings into an actual prediction.  You can see these embeddings here.

In [14]:
embedding_weights = list(model[0].parameters())[0]
print(embedding_weights.shape)
print(embedding_weights)


torch.Size([50, 8])
Parameter containing:
tensor([[ 1.8227e-01,  6.4768e-01,  9.8317e-01,  7.0750e-02,  1.9937e+00,
         -7.6557e-01,  6.4595e-01, -1.5744e+00],
        [ 1.7435e+00, -3.8688e-01,  7.2316e-02, -1.3838e+00,  1.0773e+00,
         -7.7818e-01, -3.6222e-02, -1.5022e+00],
        [ 5.9640e-01, -6.9214e-01,  7.9197e-02,  1.1759e+00, -6.4323e-01,
          2.0995e+00, -1.1213e-01, -5.1784e-01],
        [ 8.7202e-01, -2.5420e-01,  1.5559e+00,  1.4236e-01,  1.3028e+00,
         -7.6698e-01, -1.8331e-03, -6.5474e-01],
        [ 3.7451e-01,  2.8204e-01, -6.1749e-02, -1.7686e+00, -4.8501e-01,
         -2.3810e-01, -2.3741e-01,  3.3530e-01],
        [-1.8818e-01,  1.5411e+00, -9.6873e-01, -2.4956e-01, -1.8406e+00,
         -1.2305e+00,  1.0544e+00,  3.2221e-01],
        [-1.4060e-01, -6.9263e-01, -4.3914e-01, -9.5599e-01,  1.7538e+00,
          3.7769e-01, -7.1930e-01, -6.5121e-01],
        [-1.2826e+00,  2.1817e-01,  9.2549e-01,  2.0709e+00, -1.3156e-01,
          6.9829e-01, -

We can now evaluate this neural network's accuracy, including the embeddings and the learned dense layer.  

In [15]:
# Evaluation
with torch.no_grad():
    outputs = model(padded_reviews.long())
    predictions = (outputs > 0.5).float().squeeze()
    labels_tensor = torch.tensor(labels, dtype=torch.float)
    accuracy = (predictions == labels_tensor).float().mean().item()
    loss_value = criterion(outputs.squeeze(), labels_tensor).item()

print(f'Accuracy: {accuracy}')
print(f'Log-loss: {loss_value}')


Accuracy: 0.800000011920929
Log-loss: 0.4986347258090973


The accuracy is great, but there could be overfitting. It would be good to use early stopping to not overfit for a more complex data set. However, the loss is not perfect. Even though the predicted probabilities indicated a correct prediction in every case, the program did not achieve absolute confidence in each correct answer. The lack of confidence was likely due to the small amount of noise (previously discussed) in the data set. Some words that appeared in both positive and negative reviews contributed to this lack of absolute certainty.